#   Medicamentos GLP-1

Análisis completo del dataset de medicamentos GLP-1 (antagonistas del receptor GLP-1) utilizando datos reales de:
- **FDA FAERS** — Eventos adversos reportados
- **ClinicalTrials.gov** — Ensayos clínicos registrados
- **Yahoo Finance** — Precios históricos de acciones
- **Wikipedia** — Resúmenes farmacológicos

### Archivos disponibles (`data/raw/`)
| Archivo | Contenido |
|---|---|
| `drugs_overview.csv` | Marcas, fabricantes, indicaciones, aprobación FDA |
| `adverse_events.csv` | Reportes individuales de eventos adversos |
| `adverse_events_summary.csv` | Resumen estadístico por reacción |
| `clinical_trials.csv` | Ensayos clínicos registrados |
| `stock_prices.csv` | Cotizaciones bursátiles (LLY y NVO) |
| `search_trends.csv` | Tendencias de búsqueda Google |
| `wikipedia_summaries.csv` | Resúmenes de Wikipedia |
| `data_dictionary.csv` | Diccionario de datos |

**Objetivo**: Identificar patrones clínicos, comparar eficacia y seguridad entre medicamentos centrándonos en Ozempic y Mounjaro, y correlacionar hitos regulatorios con el comportamiento bursátil.

> **Dos niveles de evidencia en este notebook**
> - **Del dataset (EDA reproducible)**: todo lo que sale de los 8 CSV — stocks, FAERS, registro de ensayos.
> - **De literatura (contexto)**: los resultados de pérdida de peso (STEP 1, SURMOUNT-1, SURPASS-2). `clinical_trials.csv` solo trae el *registro* de cada ensayo, no sus resultados; estos se añaden a mano en `src/features.py` (`CLINICAL_WEIGHT_LOSS`) con su fuente y se cruzan por NCT con el dataset. Las secciones lo indican explícitamente.


## 1. Importar librerías

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Subir hasta la raíz del proyecto (donde vive src/), sea cual sea el directorio de ejecución
ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.io import load_all_datasets
from src.cleaning import clean_drugs, clean_stocks, clean_adverse_events, add_fda_in_stock_range
from src.features import (
    stock_normalized_growth, stock_growth_summary, stock_impact_at_fda, stock_impact_sensitivity,
    tabla_gravedad, demographic_profile,
    weight_trials_from_dataset, clinical_weight_loss_summary,
    HITOS_BURSATILES, AGE_LABELS,
)
from src.viz import plot_stock_comparison, plot_adverse_events, plot_weight_loss
from src.utils import print_dataset_summary, assert_columns, build_quality_report

print("Librerías e importaciones de src/ correctas")


## 2. Configurar rutas y cargar datos

In [ ]:
base_path = ROOT / "data" / "raw"
OUTPUT    = ROOT / "output"
PROCESSED = ROOT / "data" / "processed"
OUTPUT.mkdir(exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print(f" Ruta base: {base_path}")
print(f" Existe: {base_path.exists()}")
print(f" Salida gráficos: {OUTPUT}")

# Cargar todos los datasets desde src.io
datasets = load_all_datasets(base_path)

drugs   = datasets["drugs_overview"]
stocks  = datasets["stock_prices"]
ae      = datasets["adverse_events"]
aes     = datasets["adverse_events_summary"]
trials  = datasets["clinical_trials"]
trends  = datasets["search_trends"]
wiki    = datasets["wikipedia_summaries"]
ddict   = datasets["data_dictionary"]

print("\n Todos los datasets cargados correctamente")
print_dataset_summary(datasets)

## 3. Limpieza de datos y quality checks

In [ ]:
# Limpieza usando funciones de src/cleaning.py (devuelven copias: `datasets` conserva los raw)
drugs  = clean_drugs(drugs)
stocks = clean_stocks(stocks)
ae     = clean_adverse_events(ae)
drugs  = add_fda_in_stock_range(drugs, stocks)

# Validaciones (src/utils.py)
assert_columns(drugs,  ["generic_name", "brand_names", "ticker", "fda_first_approval_date", "fecha_en_rango_stock"])
assert_columns(stocks, ["ticker", "date", "close"])
assert_columns(ae,     ["brand_queried", "serious", "seriousness_death", "patient_sex", "patient_age", "country"])
print("Validaciones OK\n")

# Tabla de quality checks: filas antes/después, duplicados, nulos, fechas FDA fuera de rango, booleanos convertidos
quality = build_quality_report(
    raw={k: datasets[k] for k in ["drugs_overview", "stock_prices", "adverse_events"]},
    clean={"drugs_overview": drugs, "stock_prices": stocks, "adverse_events": ae},
    drugs_clean=drugs,
)
quality.to_csv(PROCESSED / "quality_checks.csv")
quality.T


## 4. Exploración general de los datasets

In [ ]:
print("=" * 70)
print("RESUMEN GENERAL DE TODOS LOS ARCHIVOS")
print("=" * 70)

datasets = {
    'drugs_overview'          : drugs,
    'stock_prices'            : stocks,
    'adverse_events'          : ae,
    'adverse_events_summary'  : aes,
    'clinical_trials'         : trials,
    'search_trends'           : trends,
    'wikipedia_summaries'     : wiki,
    'data_dictionary'         : ddict,
}

summary = []
for nombre, df in datasets.items():
    null_pct = df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) * 100
    summary.append({
        'Archivo'        : nombre,
        'Filas'          : f"{df.shape[0]:,}",
        'Columnas'       : df.shape[1],
        'Nulos (%)'      : f"{null_pct:.1f}%",
        'Duplicados'     : df.duplicated().sum(),
        'Cols numericas' : len(df.select_dtypes(include='number').columns),
    })

df_summary = pd.DataFrame(summary)
print(df_summary.to_string(index=False))

## 5. Análisis de medicamentos y marcas comerciales

In [ ]:
print("=" * 70)
print("MEDICAMENTOS Y MARCAS COMERCIALES")
print("=" * 70)

print(drugs[['generic_name','brand_names','manufacturer','fda_first_approval_date','is_investigational','drug_class']].to_string(index=False))

print("\n\n Por fabricante:")
print(drugs.groupby('manufacturer')[['generic_name']].count().rename(columns={'generic_name':'Total medicamentos'}).to_string())

print("\n\n🔬 Investigacionales:")
inv = drugs[drugs['is_investigational'] == True]
print(inv[['generic_name','manufacturer','drug_class']].to_string(index=False))

## 6. Stock vs aprobaciones FDA (Q1) y crecimiento por tramos (Q4)

In [ ]:
# ── Q1 · Impacto en la acción alrededor de la aprobación FDA (ventana ±30 días naturales) ──
WINDOW = 30
df_impact = stock_impact_at_fda(stocks, drugs, window_days=WINDOW)
df_impact.to_csv(PROCESSED / 'fda_stock_impact.csv', index=False)
print(f"=== IMPACTO EN ACCION POR APROBACION FDA (±{WINDOW} dias) ===\n")
print(df_impact.to_string(index=False))

fuera_de_rango = drugs[drugs['fda_first_approval_date'].notna() & drugs['ticker'].notna() & (drugs['fecha_en_rango_stock'] == False)]
if not fuera_de_rango.empty:
    print("\n[EXCLUIDOS - fecha FDA anterior al rango del stock]")
    for _, r in fuera_de_rango.iterrows():
        print(f"  {r['generic_name']:<20} | {r['ticker']} | FDA: {r['fda_first_approval_date'].date()}")

# Sensibilidad a la ventana: el resultado depende mucho de N → NO es un efecto causal de la aprobación
df_sens = stock_impact_sensitivity(stocks, drugs, windows=(10, 20, 30, 45, 60))
df_sens.to_csv(PROCESSED / 'fda_stock_impact_sensitivity.csv', index=False)
print("\n=== SENSIBILIDAD A LA VENTANA (Cambio_pct segun dias) ===")
print(df_sens.drop(columns='Fecha_FDA').to_string(index=False))
print("\n  -> Lectura descriptiva: el precio en ±N dias mezcla mercado general, resultados trimestrales,")
print("     guidance y expectativas ya descontadas. No aisla el efecto de la aprobacion.")

# ── Q4 · Crecimiento bursátil por tramos + drawdown ──
FECHA_INICIO = '2017-01-01'
FECHA_FIN    = str(stocks['date'].max().date())   # último dato disponible (el dataset llega a 2026)
df_growth = stock_growth_summary(stocks, FECHA_INICIO, FECHA_FIN, HITOS_BURSATILES)
df_growth.to_csv(PROCESSED / 'stock_growth_summary.csv')
print(f"\n=== CRECIMIENTO BURSATIL {FECHA_INICIO} -> {FECHA_FIN} ===")
print("  Hitos: MOUNJARO (dataset) y ZEPBOUND (fecha FDA 2023-11-08 añadida a mano, no está en drugs_overview)")
df_growth.T


## 7. Gráfico: Crecimiento bursátil LLY vs NVO (2017 → último dato)

In [ ]:
lly = stock_normalized_growth(stocks, 'LLY', FECHA_INICIO, FECHA_FIN)
nvo = stock_normalized_growth(stocks, 'NVO', FECHA_INICIO, FECHA_FIN)
hitos = drugs[drugs['fecha_en_rango_stock'] == True].copy()

plot_stock_comparison(
    lly, nvo, hitos,
    fecha_inicio=FECHA_INICIO,
    fecha_fin=FECHA_FIN,
    save_path=OUTPUT / 'crecimiento_stock_glp1.png'
)
print(f"Grafico guardado: {OUTPUT / 'crecimiento_stock_glp1.png'}")


## 8. Comparación de efectos adversos: OZEMPIC vs MOUNJARO (Q3)

**Cómo leer estos porcentajes.** FAERS es un sistema de notificación espontánea: *"49.8% de reportes graves"* significa que de los reportes recibidos sobre OZEMPIC, la mitad fueron marcados como graves — **no** que el 49.8% de los pacientes sufra un evento grave. Además:
- Hay sesgo de notificación (los eventos graves se reportan más; la vigilancia varía por fármaco y año).
- No se controla exposición ni volumen de uso: más reportes puede reflejar más pacientes, más tiempo en mercado o más atención mediática.
- Las ventanas de reporte del dataset son distintas para cada marca (se imprimen abajo), lo que por sí solo ya sesga la comparación.


In [ ]:
MARCAS_AE = ['OZEMPIC', 'MOUNJARO']
ozempic   = ae[ae['brand_queried'] == 'OZEMPIC'].copy()
mounjaro  = ae[ae['brand_queried'] == 'MOUNJARO'].copy()
ozempic_s  = aes[aes['generic_name'] == 'semaglutide'].copy()
mounjaro_s = aes[aes['generic_name'] == 'tirzepatide'].copy()

print(f"OZEMPIC:  {len(ozempic):,} reportes | MOUNJARO: {len(mounjaro):,} reportes\n")

# Gravedad (src/features.py) — % sobre reportes, con ventana de reporte como contexto de exposición
df_gravedad = tabla_gravedad(ae, MARCAS_AE)
df_gravedad.to_csv(PROCESSED / 'adverse_events_gravedad.csv')
print("=== COMPARACION DE GRAVEDAD (% de reportes FAERS, no incidencia) ===")
print(df_gravedad.to_string())

print("\n=== REPORTES POR AÑO (ventanas de reporte distintas por marca) ===")
sub_ae = ae[ae['brand_queried'].isin(MARCAS_AE)]
print(pd.crosstab(sub_ae['receive_date'].dt.year, sub_ae['brand_queried']).to_string())

print("\n=== TOP 8 REACCIONES MAS FRECUENTES ===")
for nombre, df_s in [('OZEMPIC', ozempic_s), ('MOUNJARO', mounjaro_s)]:
    top = df_s.nlargest(8, 'report_count')[['reaction','report_count','pct_serious','pct_hospitalization','pct_death']].copy()
    top.columns = ['Reaccion','N reportes','% Graves','% Hosp.','% Muerte']
    for c in ['% Graves','% Hosp.','% Muerte']:
        top[c] = (top[c]*100).round(1)
    print(f"\n  {nombre}")
    print(top.to_string(index=False))


## 9. Gráfico: Efectos adversos comparados

In [ ]:
plot_adverse_events(
    ozempic, mounjaro, ozempic_s, mounjaro_s, df_gravedad,
    save_path=OUTPUT / 'comparacion_efectos_adversos.png'
)
print(f"Grafico guardado: {OUTPUT / 'comparacion_efectos_adversos.png'}")

## 10. Perfil demográfico de los reportes (Q5)

Edad, sexo, país y gravedad por grupo demográfico de los reportes FAERS de OZEMPIC y MOUNJARO. Se calcula con `demographic_profile()` (`src/features.py`) y se exporta a `data/processed/demografia_*.csv`. Los porcentajes de edad se calculan solo sobre reportes con edad conocida en años (`patient_age_unit == 'Year'`).


In [ ]:
demo = demographic_profile(ae, MARCAS_AE)
for nombre, df in demo.items():
    df.to_csv(PROCESSED / f'demografia_{nombre}.csv', index=False)

print("=== EDAD ===")
print(demo['edad'].to_string(index=False))

print("\n=== SEXO (% de reportes) ===")
print(demo['sexo'].pivot(index='Sexo', columns='Medicamento', values='Pct').to_string())

print("\n=== GRUPO DE EDAD (% de reportes con edad conocida) ===")
print(demo['grupo_edad'].pivot(index='Grupo edad', columns='Medicamento', values='Pct').reindex(AGE_LABELS).to_string())

print("\n=== TOP PAISES (% de reportes) ===")
print(demo['pais'].pivot(index='Pais', columns='Medicamento', values='Pct').fillna(0).sort_values('OZEMPIC', ascending=False).to_string())

print("\n=== GRAVEDAD POR SEXO (% graves) ===")
print(demo['gravedad_sexo'].pivot(index='Sexo', columns='Medicamento', values='Graves (%)').to_string())

print("\n=== GRAVEDAD POR GRUPO DE EDAD (% graves) ===")
print(demo['gravedad_edad'].pivot(index='Grupo edad', columns='Medicamento', values='Graves (%)').reindex(AGE_LABELS).to_string())


## 11. Ensayos clínicos y pérdida de peso (Q2)

Dos bloques claramente separados:
1. **Del dataset** (`clinical_trials.csv`): qué ensayos completados sobre peso/obesidad hay registrados para semaglutide y tirzepatide (título, fase, n, fechas). ClinicalTrials.gov **no incluye resultados**.
2. **De literatura**: la pérdida de peso publicada de STEP 1, SURMOUNT-1 y SURPASS-2, definida en `CLINICAL_WEIGHT_LOSS` (`src/features.py`) con su fuente, y **cruzada por NCT** con el dataset para comprobar que cada ensayo existe en él. Se exporta a `data/processed/clinical_weight_loss_summary.csv`.


In [ ]:
# ── 1) DEL DATASET: registro de ensayos ──
weight_trials = weight_trials_from_dataset(trials)
weight_trials.to_csv(PROCESSED / 'weight_trials_dataset.csv', index=False)
print(f"=== ENSAYOS COMPLETADOS SOBRE OBESIDAD/PESO EN clinical_trials.csv: {len(weight_trials)} ===\n")
for drug, group in weight_trials.groupby('drug_query'):
    print(f"--- {drug.upper()} ({len(group)} ensayos completados) ---")
    for _, r in group.head(5).iterrows():
        print(f"  [{r['phase']}] {r['brief_title'][:75]}...")
        print(f"         {r['nct_id']} | n={r['enrollment']} | {r['start_date']} -> {r['completion_date']}")
    print()

h2h = trials[trials['nct_id'] == 'NCT03987919']
print("=== ENSAYO DIRECTO EN EL DATASET: SURPASS-2 (tirzepatide vs semaglutide) ===")
if h2h.empty:
    print("  NCT03987919 no está en clinical_trials.csv")
else:
    r = h2h.iloc[0]
    print(f"  {r['nct_id']} | {r['brief_title']}\n  Fase {r['phase']} | n={r['enrollment']} | {r['start_date']} -> {r['completion_date']} | {r['overall_status']}")

# ── 2) DE LITERATURA: resultados publicados, cruzados por NCT ──
df_clinical = clinical_weight_loss_summary(trials)
df_clinical.to_csv(PROCESSED / 'clinical_weight_loss_summary.csv', index=False)
print("\n=== RESULTADOS DE PERDIDA DE PESO (LITERATURA, no del CSV) ===")
print(df_clinical[['marca','dosis','ensayo','nct_id','N','N_dataset','duracion_semanas','perdida_peso_pct','poblacion','en_dataset']].to_string(index=False))

obes = df_clinical[df_clinical['poblacion'].str.contains('sin DM2')].set_index('marca')['perdida_peso_pct']
dif = obes['ZEPBOUND'] - obes['WEGOVY']
print(f"\n  -> En obesidad sin DM2 (dosis maxima): tirzepatide {obes['ZEPBOUND']}% vs semaglutide {obes['WEGOVY']}%")
print(f"     = {abs(dif):.1f} pp mas de perdida (~{abs(dif)/abs(obes['WEGOVY'])*100:.0f}% mas). OJO: ensayos distintos (STEP 1 vs SURMOUNT-1),")
print(f"     poblaciones y duraciones no identicas; no es una comparacion head-to-head.")


## 12. Gráfico comparativo de pérdida de peso (literatura)

In [ ]:
plot_weight_loss(df_clinical, save_path=OUTPUT / 'comparativa_perdida_peso.png')
print(f"Grafico guardado: {OUTPUT / 'comparativa_perdida_peso.png'}")


## 13. Conclusiones y hallazgos clave

In [ ]:
g = df_growth
imp = df_impact.set_index('Ticker')
grav = df_gravedad
ed = demo['edad'].set_index('Medicamento')
sx = demo['sexo'].pivot(index='Sexo', columns='Medicamento', values='Pct')

print("=" * 70)
print("  RESUMEN EJECUTIVO Y HALLAZGOS CLAVE (valores calculados en este notebook)")
print("=" * 70)
print(f"""
1. MARCAS Y FABRICANTES
   - {len(drugs)} medicamentos GLP-1 en el dataset: {(~drugs['is_investigational']).sum()} aprobados, {drugs['is_investigational'].sum()} investigacionales
   - Eli Lilly y Novo Nordisk son los unicos fabricantes con datos bursatiles (LLY, NVO)

2. BOLSA (datos hasta {FECHA_FIN})
   - Crecimiento total desde 2017: LLY {g.loc['LLY','Crecimiento_total_pct']:+.1f}% | NVO {g.loc['NVO','Crecimiento_total_pct']:+.1f}%
   - Desde MOUNJARO (may-2022): LLY {g.loc['LLY','Crecimiento_desde_MOUNJARO_pct']:+.1f}% | NVO {g.loc['NVO','Crecimiento_desde_MOUNJARO_pct']:+.1f}%
   - Desde ZEPBOUND (nov-2023): LLY {g.loc['LLY','Crecimiento_desde_ZEPBOUND_pct']:+.1f}% | NVO {g.loc['NVO','Crecimiento_desde_ZEPBOUND_pct']:+.1f}%
   - Max drawdown: NVO {g.loc['NVO','Max_drawdown_pct']}% (pico {g.loc['NVO','Drawdown_pico']} -> valle {g.loc['NVO','Drawdown_valle']}) | LLY {g.loc['LLY','Max_drawdown_pct']}%
   - Alrededor de la aprobacion FDA (±{WINDOW}d): OZEMPIC/NVO {imp.loc['NVO','Cambio_pct']:+.1f}% | MOUNJARO/LLY {imp.loc['LLY','Cambio_pct']:+.1f}%
     -> el signo cambia con la ventana (ver tabla de sensibilidad): descriptivo, NO causal.

3. EFICACIA: PERDIDA DE PESO  [LITERATURA, no del CSV]
   - Semaglutide 2.4mg (WEGOVY, STEP 1):      {obes['WEGOVY']}% de peso corporal
   - Tirzepatide 15mg (ZEPBOUND, SURMOUNT-1): {obes['ZEPBOUND']}% de peso corporal
   - SURPASS-2 (unico head-to-head, en DM2): tirzepatide 15mg -11.2% vs semaglutide 1mg -5.7%
   - El dataset aporta el registro de {len(weight_trials)} ensayos completados sobre peso, no sus resultados

4. SEGURIDAD: REPORTES FAERS  [% sobre reportes, no incidencia]
   - OZEMPIC:  {grav.loc['OZEMPIC','Graves (%)']}% de reportes graves, {grav.loc['OZEMPIC','Hospitalizacion (%)']}% con hospitalizacion (n={grav.loc['OZEMPIC','Total reportes']:,}, reportes {grav.loc['OZEMPIC','Primer reporte']} -> {grav.loc['OZEMPIC','Ultimo reporte']})
   - MOUNJARO: {grav.loc['MOUNJARO','Graves (%)']}% graves, {grav.loc['MOUNJARO','Hospitalizacion (%)']}% con hospitalizacion (n={grav.loc['MOUNJARO','Total reportes']:,}, reportes {grav.loc['MOUNJARO','Primer reporte']} -> {grav.loc['MOUNJARO','Ultimo reporte']})
   - Las ventanas de reporte NO son comparables y no hay denominador de exposicion:
     la diferencia no puede leerse como "OZEMPIC es mas peligroso"
   - MOUNJARO: los reportes mas frecuentes son errores de dosificacion, no toxicidad directa

5. PERFIL DE LOS REPORTES (Q5)
   - OZEMPIC:  edad media {ed.loc['OZEMPIC','Edad media']} a | {sx.loc['Female','OZEMPIC']}% mujeres
   - MOUNJARO: edad media {ed.loc['MOUNJARO','Edad media']} a | {sx.loc['Female','MOUNJARO']}% mujeres
   - La gravedad por grupo de edad NO sigue el mismo patron: en OZEMPIC es mayor en reportes
     jovenes (18-29) y baja con la edad; en MOUNJARO es maxima en 60-74. Poblaciones distintas.
""")
print("=" * 70)


In [ ]:
### Gráficos de conclusiones
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

DARK_BG = '#0f1117'
CARD_BG = '#1a1d27'

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor(DARK_BG)

def estilo(ax):
    ax.set_facecolor(CARD_BG)
    ax.tick_params(colors='#cccccc', labelsize=10)
    for sp in ax.spines.values(): sp.set_color('#333333')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.label.set_color('#cccccc')
    ax.xaxis.label.set_color('#cccccc')
    ax.title.set_color('white')
    ax.grid(True, axis='y', linestyle='--', alpha=0.2, color='#555555')

# ── GRÁFICO 1: Pérdida de peso ───────────────────────────────────
ax = axes[0]
estilo(ax)
# Valores desde df_clinical (literatura, ver seccion 11) — no hardcodeados aqui
_c = df_clinical
farmacos = [f"{m.capitalize()}\n({b} {d.replace('/sem', '')})" for m, b, d in zip(_c['medicamento'], _c['marca'], _c['dosis'])]
perdidas = (-_c['perdida_peso_pct']).tolist()
colores  = ['#4fc3f7' if m == 'tirzepatide' else '#ef9a9a' for m in _c['medicamento']]
ensayos  = _c['ensayo'].tolist()

bars = ax.bar(farmacos, perdidas, color=colores, alpha=0.85, width=0.55)
for bar, val, ens in zip(bars, perdidas, ensayos):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'-{val}%\n({ens})', ha='center', va='bottom',
            color='white', fontsize=8, fontweight='bold')

ax.set_title('Pérdida de peso corporal\n(literatura: STEP 1 / SURMOUNT-1 / SURPASS-2)', fontsize=11, pad=10)
ax.set_ylabel('% pérdida de peso', fontsize=10)
ax.set_ylim(0, 27)
ax.tick_params(axis='x', labelsize=8)
oz_p = mpatches.Patch(color='#ef9a9a', label='Semaglutide')
mo_p = mpatches.Patch(color='#4fc3f7', label='Tirzepatide')
ax.legend(handles=[oz_p, mo_p], facecolor='#1e1e2e', edgecolor='#444', labelcolor='white', fontsize=9)

# ── GRÁFICO 2: Perfil de seguridad ───────────────────────────────
ax = axes[1]
estilo(ax)
cats   = ['Graves', 'Muerte', 'Riesgo\nvital', 'Hospitaliz.', 'Discapacidad']
cols_g = ['Graves (%)', 'Muerte (%)', 'Riesgo vital (%)', 'Hospitalizacion (%)', 'Discapacidad (%)']
vals_oz = [df_gravedad.loc['OZEMPIC',  c] for c in cols_g]
vals_mo = [df_gravedad.loc['MOUNJARO', c] for c in cols_g]

x = np.arange(len(cats))
w = 0.35
bars_oz = ax.bar(x - w/2, vals_oz, w, label='OZEMPIC',  color='#ef9a9a', alpha=0.85)
bars_mo = ax.bar(x + w/2, vals_mo, w, label='MOUNJARO', color='#4fc3f7', alpha=0.85)

for bar in list(bars_oz) + list(bars_mo):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', color='white', fontsize=7.5)

ax.set_xticks(x)
ax.set_xticklabels(cats, fontsize=9)
ax.set_title('Perfil de seguridad\n(% sobre reportes FAERS, no incidencia)', fontsize=11, pad=10)
ax.set_ylabel('% de reportes', fontsize=10)
ax.legend(facecolor='#1e1e2e', edgecolor='#444', labelcolor='white', fontsize=9)

# ── GRÁFICO 3: Crecimiento bursátil ──────────────────────────────
ax = axes[2]
estilo(ax)
empresas = ['Eli Lilly\n(LLY)', 'Novo Nordisk\n(NVO)']
crecimientos = [df_growth.loc['LLY', 'Crecimiento_total_pct'], df_growth.loc['NVO', 'Crecimiento_total_pct']]

bars = ax.bar(empresas, crecimientos, color=['#4fc3f7', '#ef9a9a'], alpha=0.85, width=0.45)
for bar, val in zip(bars, crecimientos):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'+{val}%', ha='center', va='bottom', color='white', fontsize=12, fontweight='bold')

ax.set_title(f'Crecimiento bursátil total\n({FECHA_INICIO[:4]} → {FECHA_FIN[:4]})', fontsize=11, pad=10)
ax.set_ylabel('% crecimiento', fontsize=10)
ax.set_ylim(0, max(crecimientos) * 1.2)

fig.suptitle('Resumen de hallazgos — Medicamentos GLP-1', fontsize=14, color='white', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT / 'conclusiones_resumen.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f"Gráfico guardado: {OUTPUT / 'conclusiones_resumen.png'}")

## Conclusiones clave

**Bolsa (del dataset).** Eli Lilly multiplica por ~12 su cotización desde 2017 frente a ~2,3x de Novo Nordisk, y la brecha se abre a partir de la aprobación de MOUNJARO (may-2022). Desde entonces LLY sube y NVO cae; desde ZEPBOUND (nov-2023) NVO acumula un drawdown superior al 70% desde su pico de junio de 2024. Esto es coherente con el éxito comercial de tirzepatide, pero **es una correlación temporal**: el análisis de ±N días alrededor de cada aprobación cambia de signo según la ventana elegida y no controla mercado, resultados trimestrales ni expectativas ya descontadas.

**Eficacia (de literatura, no del dataset).** Tirzepatide 15 mg logra más pérdida de peso que semaglutide 2.4 mg en sus respectivos ensayos pivotales (-20.9% vs -14.9%) y en el único head-to-head disponible en DM2 (SURPASS-2: -11.2% vs -5.7% con semaglutide 1 mg). Son ensayos distintos con poblaciones y duraciones distintas; `clinical_trials.csv` solo confirma que estos ensayos existen y están completados.

**Seguridad (FAERS, del dataset).** OZEMPIC tiene un porcentaje de reportes graves mayor que MOUNJARO (49.8% vs 19.2% *de los reportes*). Esto **no** es incidencia ni riesgo relativo:
- FAERS es notificación espontánea, sin denominador de pacientes expuestos y con sesgo de notificación.
- Las ventanas de reporte del dataset son distintas (OZEMPIC hasta 2020; MOUNJARO concentrado en 2022, sus primeros meses en mercado), y hay reportes anteriores a la aprobación.
- Los reportes de MOUNJARO corresponden a pacientes más jóvenes (media ~49 vs ~60 años) y con más mujeres (74% vs 60%). La gravedad por grupo de edad no sigue el mismo patrón en ambos fármacos (en OZEMPIC es máxima en 18-29 y baja con la edad; en MOUNJARO es máxima en 60-74), así que la comparación global mezcla poblaciones y contextos de notificación distintos.
- Los reportes más frecuentes de MOUNJARO son errores de dosificación, no toxicidad clínica.

**Lo que sí se puede afirmar** es que el patrón de reportes es diferente y que la ventaja bursátil de Eli Lilly coincide con el ciclo de tirzepatide. Para pasar de correlación a inferencia haría falta un denominador de exposición (prescripciones) y un event study con benchmark de mercado.
